# Project Setup: Instacart Market Basket Analysis

# Data Loading and Schema Understanding

This notebook performs the initial data ingestion and structural audit of the Instacart Online Grocery Shopping dataset (6 relational tables: orders, order_products__prior, order_products__train, products, aisles, departments), ahead of exploratory data analysis (EDA) and Market Basket Analysis (MBA).

### Objectives
- Load all Instacart source tables
- Inspect row/column structure
- Understand how the tables relate to each other (shared keys: `order_id`, `product_id`, `aisle_id`, `department_id`)
- Identify the grain of each table
- Validate data types, missing values, duplicates, and categorical distributions
- Prepare a clean, well-understood foundation for EDA and Market Basket Analysis

## 1.1 Import Libraries and Load Datasets

### Objective
Import the required libraries, configure display settings for readability, and load all six raw Instacart CSV files into memory. Datasets are stored in a dictionary (`datasets`) so that later steps can iterate over every table without repeating code.

In [ ]:
import pandas as pd
import numpy as np  
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
#Display settings for better visualization
# ==========================================
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', '{:,.2f}'.format)
print("Libraries imported successfully.")

# ==========================================
# File paths to the raw datasets
# ==========================================
orders_path = "../data/raw/orders.csv"
prior_path = "../data/raw/order_products__prior.csv"
train_path = "../data/raw/order_products__train.csv"
products_path = "../data/raw/products.csv"
aisles_path = "../data/raw/aisles.csv"
departments_path = "../data/raw/departments.csv"

# ==========================================
#Load all sources tables
# ==========================================
orders = pd.read_csv(orders_path)
order_products_prior = pd.read_csv(prior_path)
order_products_train = pd.read_csv(train_path)
products = pd.read_csv(products_path)
aisles = pd.read_csv(aisles_path)
departments = pd.read_csv(departments_path)

#==================================================================
# Store datasets in a dictionary for iteration across the notebook
#==================================================================
datasets = {
    "Orders": orders,
    "Order Products Prior": order_products_prior,
    "Order Products Train": order_products_train,
    "Products": products,
    "Aisles": aisles,
    "Departments": departments
}
print("All Datasets loaded successfully.")


# Initial Data Exploration

## 2.1 Dataset Preview

### Objective
Preview the first few rows of each dataset to sanity-check the data visually before running deeper checks.

In [ ]:
#==========================================
#Preview top rows of each dataset
#==========================================

for name, df in datasets.items():
    print(f"preview of {name}")
    display(df.head())

### Observation
- Row-level content matches the expected schema for each table (e.g. `orders` shows one row per order with sequencing fields like `order_number` and `days_since_prior_order`).
- `eval_set` in `orders` confirms the prior/train/test split used by Instacart's original competition design.

## 2.2 Dataset Shape

### Objective
Check the number of rows and columns in each dataset to understand its scale and grain before further inspection.

In [ ]:
#==========================================
#shape of the each datasets
#==========================================
for name, df in datasets.items():
    print(f"{name} shape: {df.shape}")

### Observation
- `order_products_prior` (32.4M rows) is by far the largest table, representing the full history of prior purchases, while `order_products_train` (1.4M rows) is a much smaller labeled subset held out for prediction tasks.
- `orders` (3.42M rows) sits at the order grain, one row per order.
- `products`, `aisles`, and `departments` are small lookup/dimension tables (49,688 / 134 / 21 rows respectively), each keyed by their own ID.

## 2.3 Column Names

### Objective
Inspect the column structure of each dataset to identify the join keys that link the tables together and confirm the grain of each one.

In [29]:
# ==========================================
#Columns in each dataset
# ==========================================

for name, df in datasets.items():
    print(f"Columns in {name}: {list(df.columns)}\n")


Columns in Orders: ['order_id', 'user_id', 'eval_set', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order']

Columns in Order Products Prior: ['order_id', 'product_id', 'add_to_cart_order', 'reordered']

Columns in Order Products Train: ['order_id', 'product_id', 'add_to_cart_order', 'reordered']

Columns in Products: ['product_id', 'product_name', 'aisle_id', 'department_id']

Columns in Aisles: ['aisle_id', 'aisle']

Columns in Departments: ['department_id', 'department']



### Observation
- `order_id` links `orders` to `order_products_prior` / `order_products_train`.
- `product_id` links the order-product tables to `products`.
- `aisle_id` / `department_id` appear in `products`, `aisles`, `departments` → link products to their category hierarchy.
- This confirms a classic star-schema layout: `orders` and `order_products_*` are fact tables, while `products`, `aisles`, and `departments` are dimension tables.

## 2.4 Data Types

### Objective
Inspect the data type of each column across all datasets to verify that every variable is stored in the correct format before performing analysis.

In [ ]:
# ==========================================
#Inspect the data types of each dataset
# ==========================================
for name, df in datasets.items():
    print(f"{name} data types:")
    print(df.dtypes,"\n")

### Observation
- Most columns contain whole numerical values such as identifiers and order-related information.
- The `days_since_prior_order` column may contain decimal values or missing (`NaN`) values.
- Text-based columns include `eval_set`, `product_name`, `aisle`, and `department`.
- No obvious data type inconsistencies were observed at this stage.

## 2.5 Grain, Primary Keys & Foreign Keys

### Objective

Identify the grain (level of detail), primary keys (PK), and foreign keys (FK) of each dataset to understand how the tables are related before performing joins.

| Dataset              | Grain (One Row = ...)              | Primary Key (PK)                            | Foreign Key (FK)            |
| -------------------- | ---------------------------------- | ------------------------------------------- | --------------------------- |
| Orders               | One order                          | `order_id`                                  | `user_id`                   |
| Order Products Prior | One product within one prior order | (`order_id`, `product_id`) *(Composite PK)* | `order_id`, `product_id`    |
| Order Products Train | One product within one train order | (`order_id`, `product_id`) *(Composite PK)* | `order_id`, `product_id`    |
| Products             | One product                        | `product_id`                                | `aisle_id`, `department_id` |
| Aisles               | One aisle                          | `aisle_id`                                  | —                           |
| Departments          | One department                     | `department_id`                             | —                           |

## 2.6 Missing Values

### Objective

Identify missing values across all datasets to evaluate data quality and determine whether any missing values require cleaning or represent meaningful business information.

In [ ]:
#==========================================
# Missing Values Analysis
#==========================================

for name, df in datasets.items():
    print(f"\n{name}")
    print(df.isnull().sum())

### Observation

- Only the `days_since_prior_order` column in the **Orders** dataset contains missing values (206,209 records, approximately **6.03%**).
- All other columns across the six datasets contain no missing values.
- The missing values appear to be limited to a single business-related attribute rather than indicating widespread data quality issues.

## 2.7 Duplicate Rows

### Objective

Identify duplicate records across all datasets to ensure data integrity before performing analysis.

In [ ]:
#==========================================
# Duplicate Analysis
#==========================================

for name, df in datasets.items():
    duplicate_count = df.duplicated().sum()
    duplicate_percentage = df.duplicated().mean() * 100


    print(f"\n{name}")
    print(f"Duplicate Rows: {duplicate_count}")
    print(f"Duplicate Percentage: {duplicate_percentage:.2f}%")

### Observation

- No duplicate rows were found in any of the six datasets.
- This indicates that there are no exact duplicate records at the row level.
- Further validation is still required to check for missing values, invalid values, and relationship consistency between tables.

## 2.8 Categorical Validation

### Objective

Explore categorical columns to understand the distribution of categories and validate expected values.

In [ ]:
#==========================================
# Categorical Value Validation
#==========================================

categorical_columns = {
    "Orders - eval_set": orders["eval_set"],
    "Order Products Prior - reordered": order_products_prior["reordered"],
    "Order Products Train - reordered": order_products_train["reordered"]
}

for name, column in categorical_columns.items():
    print("=" * 50)
    print(name)
    print("=" * 50)
    print(column.value_counts())
    print()

### Observation
- The `eval_set` column contains the expected categories: `prior`, `train`, and `test`.
- The `prior` dataset contains the majority of orders, indicating that most records represent customers' historical purchases.
- The `reordered` column contains only valid binary values (`0` and `1`) in both prior and train datasets.
- Reordered products (`1`) occur more frequently than non-reordered products (`0`), suggesting that customers often repurchase previously bought items.

## 2.9 Dataset Information

### Objective

Inspect the structure of each dataset, including the number of rows, columns, data types, non-null values, and memory usage.

In [ ]:
for name, df in datasets.items():
    print("=" * 60)
    print(name)
    print("=" * 60)
    df.info()
    print()


### Observation

- Dataset structures were successfully verified.
- Most columns have appropriate data types (`int64` and `object`/`str`).
- Only `days_since_prior_order` contains missing values, while all other columns are complete.
- Memory usage is reasonable for the size of each dataset.

## 2.10 Statistical Summary

### Objective

Generate summary statistics for numerical columns to understand their distribution, central tendency, and variability.

In [ ]:
for name, df in datasets.items():
    print("=" * 60)
    print(name)
    print("=" * 60)
    display(df.describe())
    print()

### Observation

- The statistical summary confirms that all numerical columns contain valid values.
- Customers placed an average of approximately 17 orders.
- Orders are typically placed around 1 PM (`order_hour_of_day` mean ≈ 13.45).
- Customers reorder after approximately 11 days on average.
- The dataset contains customers with up to 100 orders, indicating varying purchasing behavior.

### Next Steps
With schema, types, missing values, duplicates, and category distributions validated, the notebook is ready to proceed to exploratory data analysis (EDA) and feature engineering for Market Basket Analysis.

# Summary

### Key Findings

- Successfully loaded and validated all six datasets.
- Identified dataset grain, primary keys, and foreign keys.
- Verified that only `days_since_prior_order` contains missing values.
- No duplicate rows were found.
- Validated categorical columns and confirmed expected values.
- Generated statistical summaries to understand the overall data distribution.

### Conclusion

The datasets are well-structured and ready for data cleaning, feature engineering, and business analysis in the next milestone.

# 3 Data Cleaning & Preparation

## 3.1 Validate Primary Keys

### Objective

Verify that the identified primary keys uniquely identify each record and confirm the integrity of relationships before merging datasets.

### Observation

- `order_id` is unique in the Orders dataset and `product_id` is unique in the Products dataset.
- No duplicate combinations of (`order_id`, `product_id`) were found in either the Prior or Train datasets.
- The primary keys and composite primary keys are valid, indicating that the datasets maintain referential integrity and are safe to use for joins.

In [ ]:
# Orders primary key
print("Orders - order_id unique:",
      orders["order_id"].is_unique)

# Products primary key
print("Products - product_id unique:",
      products["product_id"].is_unique)

# Composite primary key (Prior)
prior_duplicates = order_products_prior.duplicated(
    subset=["order_id", "product_id"]
).sum()

print("Prior composite PK duplicates:", prior_duplicates)

# Composite primary key (Train)
train_duplicates = order_products_train.duplicated(
    subset=["order_id", "product_id"]
).sum()

print("Train composite PK duplicates:", train_duplicates)

## 3.2 Missing Value Treatment

### Objective

Evaluate the identified missing values and determine the appropriate treatment based on their business meaning to preserve data integrity.

### Observation

- Missing values exist only in the `days_since_prior_order` column.
- These missing values represent customers' first orders, where no previous order exists.
- Since the missing values are meaningful and expected, they were retained without imputation or removal.

### Decision

No changes were made to the missing values because they represent valid business information rather than data quality issues. Filling or removing these values would distort customer behavior analysis.

## 3.3 Create Analysis-Ready Dataset

### Objective

Combine the transactional and reference datasets using their primary and foreign key relationships to create a single analysis-ready dataset for customer behavior and market basket analysis.

### Observation

- Successfully merged the `Order Products Prior` and `Orders` datasets using `order_id`.
- The row count remained unchanged, confirming that no records were lost or duplicated during the merge.
- Order-level information has been successfully added to each product transaction.

In [ ]:
# Merge prior order products with orders
instacart_df = order_products_prior.merge(
    orders,
    on="order_id",
    how="left"
)

print("Merged Shape:", instacart_df.shape)

print("Expected Rows:", len(order_products_prior))
print("Actual Rows:", len(instacart_df))
display(instacart_df.head())

### Observation

- Successfully merged the Products dataset using `product_id`.
- Product names, aisle IDs, and department IDs were added to each transaction.
- The row count remained unchanged, confirming that the merge preserved all transaction records.

In [ ]:
# Merge product information
instacart_df = instacart_df.merge(
    products,
    on="product_id",
    how="left"
)

print("Merged Shape:", instacart_df.shape)
print("Expected Rows:", len(order_products_prior))
print("Actual Rows:", len(instacart_df))
display(instacart_df.head())

In [ ]:
# Merge aisle information
instacart_df = instacart_df.merge(
    aisles,
    on="aisle_id",
    how="left"
)

print("Merged Shape:", instacart_df.shape)
display(instacart_df.head())

In [ ]:
instacart_df = instacart_df.merge(
    departments,
    on="department_id",
    how="left"
)
print("Merged Shape:", instacart_df.shape)
display(instacart_df.head())

### Observation

- Successfully merged the Orders, Products, Aisles, and Departments datasets with the transaction data.
- The final analysis-ready dataset contains **32,434,489 rows** and **15 columns**.
- Row count remained unchanged after every merge, confirming that all joins preserved the transactional records.
- The dataset now includes customer, order, product, aisle, and department information required for business analysis.

## 3.4 Post-Merge Validation

### Objective

Validate the merged dataset to ensure no records were lost or duplicated and verify that key descriptive fields were successfully populated after all joins.

In [ ]:
print(instacart_df.shape)

print("\nMissing Values:")
print(instacart_df[
    ["product_name", "aisle", "department"]
].isnull().sum())

### Observation

- The merged dataset maintained the expected number of rows.
- No missing values were introduced in the descriptive columns (`product_name`, `aisle`, and `department`).
- The dataset passed all validation checks and is ready for exploratory analysis.